## 0. Disclaimer
This notebook documents the ETL pipeline that prepares the project database used by the Flask application. It explains the data sources, how the datasets are aligned, which normalization rules are applied, and what is loaded into SQLite.

The scripts for database initialization and seeding are stored in the repository's **scripts** directory, and they serve as the single source of truth. See **scripts/init_db.py** for schema creation and **scripts/seed/seed_db.py** for the seeding orchestrator. Per-table seeders live in scripts/seed.

## 1. Overview
The goal of this pipeline is to build a clean relational dataset for a web project that presents cat breeds, supports search by breed name, and enables filtering in the breeds overview page. The resulting SQLite database is served by a Flask app with Jinja templates and HTML/CSS styling.

Inputs are two JSON datasets. The primary dataset comes from The Cat API and contains the stable breed identifier, descriptive fields, ratings, and a featured image URL. The secondary dataset comes from Wikidata and provides German name, German description, and a German Wikipedia link.

Output is a SQLite database cats.db containing the following tables:
* cat_breeds,
* breed_alt_names,
* breed_de_info,
* temperament, and the junction table breed_temperament.

Images are stored as featured_image_url in cat_breeds and additional gallery images are loaded at runtime from The Cat API.

## 2. Data Sources
**The Cat API** is the primary data source. It provides structured breed records with a stable id, many numeric rating fields, and an image object that can be used as a featured image. Because it already contains a unique identifier, it is suitable as the canonical key for relational storage.

**Wikidata** is the secondary data source. It was used to extract German breed name and German Wikipedia information via the Wikidata Query Service. Coverage is incomplete compared to The Cat API dataset and there is no identifier that could be used as a key, so an additional data consolidation step is required before loading the German data into the relational DB.



## 3. Methodology
#### Identifier strategy
The Cat API **"id"** field is used as the canonical primary key **breed_id**. All dependent tables reference this key via foreign keys.

#### Cross-source mapping
The Wikidata dataset did not contain an "id". To enable table joins, a "cat API id" field was added using a ChatGPT prompt (more information below). The results were reviewed manually, and unmatched German entries were excluded from loading because they could not be linked to the main table.

#### Normalization and relational design
* Temperament is stored in third normal form by extracting unique traits into the temperament table and connecting breeds to traits through the junction table breed_temperament.
* Alternative names are stored in breed_alt_names table as a many-to-one relationship to cat_breeds (main table).
* German content is stored in breed_de_info as a one-to-one extension table keyed by breed_id.

Wikidata extraction was performed using the Wikidata Query Service. The SPARQL query below returns German Wikipedia entries that are categorized as cat breeds, including the German label and the German Wikipedia URL. The result was exported as JSON and stored as german_cat_breed_info_raw.json in the notebooks directory.

## 4. Data Extraction
### 4.1 TheCatApi
Data was requested from API using requests library, the output is stored as **cat_api_full.json** file in notebooks directory

In [22]:
import json
import requests
from pathlib import Path

# Run this notebook from the notebooks directory for predictable relative paths.
# If your working directory differs, adjust PROJECT_ROOT accordingly.
PROJECT_ROOT = Path.cwd().parent

# Common paths used in this notebook
NOTEBOOKS_DIR = PROJECT_ROOT / "notebooks"
RAW_CAT_API_JSON = NOTEBOOKS_DIR / "cat_api_full.json"
RAW_WIKIDATA_JSON = NOTEBOOKS_DIR / "wiki_data_raw.json"
MAPPED_WIKIDATA_JSON = NOTEBOOKS_DIR / "german_cat_breed_info.json"

In [23]:
CAT_API_URL = "https://api.thecatapi.com/v1/breeds"

response = requests.get(CAT_API_URL, timeout=15)
response.raise_for_status()

breeds_data = response.json()

print(f"Fetched {len(breeds_data)} breeds from The Cat API")

Fetched 67 breeds from The Cat API


In [24]:
output_path = RAW_CAT_API_JSON

with open(output_path, "w", encoding="utf-8") as f:
    json.dump(breeds_data, f, ensure_ascii=False, indent=2)

print(f"Dataset saved to {output_path.resolve()}")

Dataset saved to C:\Users\dkube\Projects\Katzenpedia\project-2\notebooks\cat_api_full.json


### 4.2 WikiData
Wikidata extraction was performed using the Wikidata Query Service. The SPARQL query below returns German Wikipedia entries that are categorized as cat breeds, including the German label and the German Wikipedia URL. The result was exported as json and stored as **german_cat_breed_info_raw.json** in the notebooks directory.

In [ ]:
# SPARQL Query
"""
PREFIX wd: <http://www.wikidata.org/entity/>
PREFIX wdt: <http://www.wikidata.org/prop/direct/>
PREFIX schema: <http://schema.org/>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>

SELECT
  ?breedName
  ?wikiTitle
  ?wikiUrl
  (SAMPLE(?imageUrl) AS ?imageUrl)
WHERE {
  # Cat breeds
  ?item wdt:P31/wdt:P279* wd:Q43577 .  # Q43577 = cat breed

  ?wikiUrl schema:about ?item ;
           schema:isPartOf <https://de.wikipedia.org/> .

  ?item rdfs:label ?breedName .
  FILTER(LANG(?breedName) = "de")

  BIND(STRAFTER(STR(?wikiUrl), "https://de.wikipedia.org/wiki/") AS ?wikiTitle)
}
GROUP BY ?breedName ?wikiTitle ?wikiUrl
ORDER BY ?breedName
"""

In [21]:
# Expected files for the remainder of the pipeline
# RAW_WIKIDATA_JSON: exported JSON from the Wikidata Query Service
# MAPPED_WIKIDATA_JSON: post-processed JSON with an added catApiId field used for joins
print("Raw Wikidata JSON:", RAW_WIKIDATA_JSON)
print("Mapped Wikidata JSON:", MAPPED_WIKIDATA_JSON)

Raw Wikidata JSON: C:\Users\dkube\Projects\Katzenpedia\project-2\notebooks\wiki_data_raw.json
Mapped Wikidata JSON: C:\Users\dkube\Projects\Katzenpedia\project-2\notebooks\german_cat_breed_info.json


## 5. Transform
The transformation step is divided into two stages.

### 5.1 Transform Stage One - data consolidation
Using chatGPT prompt it aligns the **wiki_data_raw** dataset with the **cat_api_full** dataset by adding a unique **"id"** from **catAPI** to each German record when a confident match is found, ensuring relational integration during the DB seeding/loading step.

**Inputs:**
- The Cat API dataset with stable id in the id field.
- The wiki_data_raw dataset with German fields: breedName, wikiUrl, and optional imageUrl.

**Output:** A cleaned German dataset that includes id from catAPI. The finalized file is stored as **german_cat_breed_info.json**


### Used ChatGPT prompt:

You are given TWO uploaded JSON files:

* The first uploaded file is **file1** (German cat breed data)
* The second uploaded file is **file2** (English cat breed data)

Read both uploaded files directly. Do NOT ask the user to paste their contents.

---

### Your task

Return EXACTLY ONE output: a JSON object.

The output JSON must contain:

* ALL properties from **file1**, unchanged
* PLUS a new property "Description" (always present)
* PLUS "id" copied from file2.id ONLY if both files describe the SAME cat breed

Do NOT include any other properties from file2.
Do NOT include explanations or extra text.

---

### Step 1 — Determine if it is the same breed (German ↔ English only)

Treat the breeds as the SAME only if **at least two** of the following signals agree:

1. **Wikipedia alignment (strong signal)**

   * file1.wikiUrl / file1.wikiTitle (German Wikipedia)
   * file2.wikipedia_url (English Wikipedia)
     If both clearly refer to the same breed topic, count as a match signal.

2. **Name translation / synonym matching**

   * Compare file1.breedName (German) with file2.name (English)
   * Handle common patterns:

     * German “-katze” forms ↔ English breed names
       (e.g., *Abessinierkatze* ↔ *Abyssinian*)
     * Spacing or hyphen differences
       (e.g., *Britisch Kurzhaar* ↔ *British Shorthair*)

3. **URL keyword consistency (optional)**

   * If both Wikipedia URLs contain unmistakable breed name tokens, count as another signal.

If unsure, treat as **NOT the same breed**.

---

### Step 2 — Generate "Description" (always)

Create "Description" in **German**, even if the breeds do not match.

Rules:

* 2–4 sentences
* Describe temperament, appearance, and care/activity level
* Prefer information implied by file1.wikiTitle or file1.wikiUrl
* Paraphrase in your own words — do NOT copy Wikipedia text
* If Wikipedia data is unclear, generate a cautious, general breed description

---

### Step 3 — Output format (VERY IMPORTANT)

* Output ONLY the final JSON
* Wrap it in a single fenced code block labeled
json
* No text before or after the code block

#### Output structure:

* MATCH ⇒ `{ ...file1, "id": file2.id, "Description": "<German text>" }`
* NO MATCH ⇒ `{ ...file1, "Description": "<German text>" }`

---

Now read the uploaded files and produce the final JSON.


#### First output: https://chatgpt.com/s/t_6975c728e7f081919356ac451d874838
The first output included some cat breeds that were present in the Wikidata dataset but not the Cat API dataset. Since these entries were not matched, they could not be enriched with a corresponding id. Consequently, the output could not be converted into a clean, downloadable JSON file.

### Second prompt:
Clean the data by removing duplicates and any objects that do not contain an id field. Then convert the result into a valid, downloadable JSON file.

#### Second output: https://chatgpt.com/s/t_6974e02ec1408191926d7ce1de16a1d1
No further adjustments were needed for the second output. The wiki_data dataset was successfully enriched with the cat API "ids", along with the generated German descriptions. The resulting dataset was downloaded and stored in notebooks directory as: **german_cat_breed_info.json**

The total AI processing time was 23 minutes: 12 minutes for the initial output and 11 minutes for the validation request.

## 5.2 Transform Stage Two - data normalization

In second stage normalization rules required by the final ER diagram were applied.
This includes:
* Parsing range fields into min and max columns,
* splitting comma separated fields into lookup tables,
* trimming strings,
* converting empty strings to NULL.

All normalization functions can be found in: **/scripts/seed/utilities.py**

**Disclaimer**: the normalization scripts were imported and used within the DB seeding/loading scripts. The normalization took place right before loading.

In [29]:
def clean_text(value):
    """
    Trim strings.
    Convert empty / whitespace-only strings to None.
    """
    if value is None:
        return None
    value = str(value).strip()
    return value if value else None


def split_csv_field(value):
    """
    Split comma-separated strings into a list of cleaned values
    Returns an empty list if input is missing or empty
    """
    if not value:
        return []
    parts = value.split(",")
    return [p.strip() for p in parts if p.strip()]


# ----------- ----------------
# Weight and age parsing Helpers
# ---------------------------

def parse_range(value):
    """
    Parse ranged values like metric_weight or life_span e.g.: 3 - 5
    Returns (min, max) or (None, None)
    """
    if not value:
        return None, None

    numbers = re.findall(r"\d+", str(value))
    if not numbers:
        return None, None

    if len(numbers) == 1:
        num = int(numbers[0])
        return num, num

    low, high = int(numbers[0]), int(numbers[1])
    return (low, high) if low <= high else (high, low)

# ---------------------------
# Image Helper
# ---------------------------
def get_featured_image_from_cat_api(breed):
    """
    Extract image.url from Cat API breed object.
    """
    image = breed.get("image")
    if not image:
        return None
    return clean_text(image.get("url"))

## 6. Load and Seed
Before loading any data, the SQLite schema is created based on the ER diagram. The schema initialization is implemented in **scripts/init_db.py**. Running this script creates or updates the database file at **instance/cats.db**. The schema includes primary keys, foreign keys with cascading deletes.

The seeding process is split into multiple independent Python scripts. Each script seeds one logical part of the database and can be executed in two ways:
* Standalone execution when you want to reseed only one table or one part of the dataset.
* Orchestrated execution when you want to rebuild the entire database in the correct order.

The seeding scripts are located in **scripts/seed**. <br> The orchestrator script **scripts/seed/seed_db.py** runs all seeding steps using one shared database connection. Using a shared connection ensures consistent transaction. Each seeding script is designed so it can accept an existing connection from the orchestrator or create and close its own connection when executed standalone

Seeding order in the n the orchestrator **seed_db.py**
1. cat_breeds is seeded from the cleaned The Cat API dataset. This table is seeded first because it provides the primary key breed_id that all dependent tables reference.
2. breed_alt_names is seeded next from the alt_names field. This is a one-to-many table keyed by the compound primary key (breed_id, alt_name).
3. temperament and breed_temperament are seeded afterwards. Unique temperament traits are inserted into the temperament lookup table, and the many-to-many relationships between breeds and traits are inserted into breed_temperament.
4. breed_de_info is seeded last from german_cat_breed_info.json. This table depends on an existing breed_id and therefore must run after cat_breeds.


Tables that represent lookup or relationship data use INSERT OR IGNORE to prevent duplicate rows when a script is run multiple times. The main table cat_breeds uses an upsert strategy with ON CONFLICT DO UPDATE so that running the seed again updates existing records rather than failing or creating duplicates. This approach allows the dataset to be refreshed after changes to transformation rules or source files, while keeping the database consistent.

#### How to run the seeding
Recommended approach is to run the orchestrator **seed_db.py** from the project root to avoid import and path issues. If scripts is a Python package, use the module form: <br>
python -m scripts.seed.seed_db

If you only want to reseed a single table, you can run the corresponding script directly, but full rebuilds should always use the orchestrator to guarantee correct order and consistent commits.

## 7. Validation
The SQLite database is stored locally at: **project-2\instance\cats.db**.

I did not perform extensive automated validation during the development phase due to the size of the database. The most populated main table, cat_breeds, contains only 67 entries; the other tables are far less populated. Therefore, a manual lookup of the tables content with the DB Browser app was sufficient.

#### In order for the documentation to be completed, I retrospectively designed a simple validation script

In [34]:
from pathlib import Path
import sqlite3

PROJECT_ROOT = Path.cwd().resolve().parent
DB_PATH = PROJECT_ROOT / "instance" / "cats.db"

if not DB_PATH.exists():
    raise FileNotFoundError(f"Database not found: {DB_PATH}")

conn = sqlite3.connect(DB_PATH)
cur = conn.cursor()

tables = [r[0] for r in cur.execute(
    "SELECT name FROM sqlite_master WHERE type='table' ORDER BY name;"
).fetchall()]

print(f"DB: {DB_PATH}\n")
print("Tables found:")
print(", ".join(tables) + "\n")

expected = {"cat_breeds", "breed_alt_names", "breed_de_info", "temperament", "breed_temperament"}
if expected.issubset(tables):
    print("All expected tables exist.")

conn.close()

DB: C:\Users\dkube\Projects\Katzenpedia\project-2\instance\cats.db

Tables found:
breed_alt_names, breed_de_info, breed_temperament, cat_breeds, sqlite_sequence, temperament

All expected tables exist.
